In [1]:
import pandas as pd
import numpy as np

In [ ]:
# Cargar datos
df = pd.read_csv('./data/US_Accidents_2022.csv')
total_records = len(df)

print("="*70)
print(" ANÁLISIS DE CALIDAD DE DATOS")
print("="*70)
print(f"Total de registros: {total_records:,}\n")

 ANÁLISIS DE CALIDAD DE DATOS
Total de registros: 1,762,449



In [39]:
# 1. Valores nulos
print("1. VALORES NULOS POR COLUMNA:")
null_counts = df.isnull().sum()
null_percentage = (null_counts / total_records * 100).round(2)
null_summary = pd.DataFrame({
    'Columna': null_counts.index,
    'Nulos': null_counts.values,
    'Porcentaje': null_percentage.values
})
null_summary = null_summary[null_summary['Nulos'] > 0].sort_values('Porcentaje', ascending=False)
print(null_summary.head(15).to_string(index=False))

1. VALORES NULOS POR COLUMNA:
              Columna  Nulos  Porcentaje
              End_Lat 236182       13.40
              End_Lng 236182       13.40
    Precipitation(in)  64475        3.66
        Wind_Chill(F)  54508        3.09
       Wind_Direction  48385        2.75
      Wind_Speed(mph)  48377        2.74
       Visibility(mi)  41953        2.38
          Humidity(%)  41107        2.33
    Weather_Condition  38688        2.20
       Temperature(F)  38718        2.20
         Pressure(in)  33134        1.88
    Weather_Timestamp  29595        1.68
       Sunrise_Sunset  14896        0.85
    Nautical_Twilight  14896        0.85
Astronomical_Twilight  14896        0.85


In [40]:
# 2. Duplicados
duplicates = df.duplicated(subset=['ID']).sum()
duplicate_pct = (duplicates / total_records * 100).round(2)
print(f"\n2. DUPLICADOS:")
print(f"   Registros duplicados: {duplicates:,} ({duplicate_pct}%)")


2. DUPLICADOS:
   Registros duplicados: 0 (0.0%)


In [41]:
# 3. Fechas inválidas
df['Start_Time'] = pd.to_datetime(df['Start_Time'], errors='coerce')
df['Year'] = df['Start_Time'].dt.year
invalid_dates = df['Start_Time'].isnull().sum()
non_2022 = ((df['Year'] != 2022) & (df['Year'].notna())).sum()
print(f"\n3. FECHAS:")
print(f"   Fechas no parseables: {invalid_dates:,}")
print(f"   Registros fuera de 2022: {non_2022:,}")


3. FECHAS:
   Fechas no parseables: 0
   Registros fuera de 2022: 0


In [42]:
# 4. Coordenadas fuera de rango USA continental
invalid_coords = ((df['Start_Lat'] < 24) | (df['Start_Lat'] > 49) | 
                  (df['Start_Lng'] < -125) | (df['Start_Lng'] > -66)).sum()
print(f"\n4. COORDENADAS:")
print(f"   Coordenadas fuera de USA continental: {invalid_coords:,}")


4. COORDENADAS:
   Coordenadas fuera de USA continental: 3


In [43]:
# 5. Severidad fuera de rango
invalid_severity = (~df['Severity'].isin([1,2,3,4])).sum()
print(f"\n5. SEVERIDAD:")
print(f"   Valores fuera de rango (1-4): {invalid_severity:,}")



5. SEVERIDAD:
   Valores fuera de rango (1-4): 0


In [44]:
# CALCULAR RUIDO TOTAL
total_noise = duplicates + invalid_dates + invalid_coords + invalid_severity + non_2022
noise_percentage = (total_noise / total_records * 100).round(2)

print(f"   Total de registros problemáticos: {total_noise:,}")
print(f"   Porcentaje de ruido: {noise_percentage}%")

   Total de registros problemáticos: 3
   Porcentaje de ruido: 0.0%


## Limpieza de datos

In [61]:
# PASO 2: Eliminar duplicados por ID
print("\nPASO 2: Eliminar duplicados")
antes_duplicados = len(df)
df = df.drop_duplicates(subset=['ID'], keep='first')
print(f"    Duplicados eliminados: {antes_duplicados - len(df):,}")
print(f"    Registros restantes: {len(df):,}")


PASO 2: Eliminar duplicados
    Duplicados eliminados: 0
    Registros restantes: 1,762,452


In [62]:
# PASO 3: Validar y filtrar coordenadas GPS
print("\nPASO 3: Validar coordenadas GPS (USA continental)")
antes_coords = len(df)
df = df[(df['Start_Lat'].between(24, 49)) & 
        (df['Start_Lng'].between(-125, -66))]
print(f"    Registros con coordenadas inválidas: {antes_coords - len(df):,}")
print(f"    Registros restantes: {len(df):,}")


PASO 3: Validar coordenadas GPS (USA continental)
    Registros con coordenadas inválidas: 3
    Registros restantes: 1,762,449


In [63]:
# PASO 4: Eliminar registros sin coordenadas
print("\nPASO 4: Eliminar registros sin coordenadas")
antes_null_coords = len(df)
df = df.dropna(subset=['Start_Lat', 'Start_Lng'])
print(f"    Registros sin coordenadas: {antes_null_coords - len(df):,}")
print(f"    Registros restantes: {len(df):,}")


PASO 4: Eliminar registros sin coordenadas
    Registros sin coordenadas: 0
    Registros restantes: 1,762,449


In [64]:
# PASO 5: Validar severidad
print("\nPASO 5: Validar severidad (1-4)")
antes_severity = len(df)
df = df[df['Severity'].isin([1,2,3,4])]
print(f"    Registros con severidad inválida: {antes_severity - len(df):,}")
print(f"    Registros restantes: {len(df):,}")


PASO 5: Validar severidad (1-4)
    Registros con severidad inválida: 0
    Registros restantes: 1,762,449


In [65]:
# PASO 6: Validar consistencia temporal
print("\nPASO 6: Validar consistencia temporal (Start_Time < End_Time)")
df['End_Time'] = pd.to_datetime(df['End_Time'], errors='coerce')
antes_time = len(df)
df = df[(df['End_Time'] >= df['Start_Time']) | df['End_Time'].isna()]
print(f"    Registros inconsistentes: {antes_time - len(df):,}")
print(f"    Registros restantes: {len(df):,}")


PASO 6: Validar consistencia temporal (Start_Time < End_Time)
    Registros inconsistentes: 0
    Registros restantes: 1,762,449


In [66]:
# PASO 7: Imputar valores nulos climáticos con mediana por estado
print("\nPASO 7: Imputar valores climáticos faltantes (mediana por estado)")
climate_cols = ['Temperature(F)', 'Humidity(%)', 'Pressure(in)', 
                'Visibility(mi)', 'Wind_Speed(mph)']
for col in climate_cols:
    nulls_antes = df[col].isnull().sum()
    df[col] = df.groupby('State')[col].transform(
        lambda x: x.fillna(x.median())
    )
    nulls_despues = df[col].isnull().sum()
    print(f"    {col}: {nulls_antes:,} → {nulls_despues:,} nulos")


PASO 7: Imputar valores climáticos faltantes (mediana por estado)
    Temperature(F): 38,718 → 0 nulos
    Humidity(%): 41,107 → 0 nulos
    Pressure(in): 33,134 → 0 nulos
    Visibility(mi): 41,953 → 0 nulos
    Wind_Speed(mph): 48,377 → 0 nulos


In [67]:
# PASO 8: Imputar End_Lat/End_Lng con Start_Lat/Start_Lng
print("\nPASO 8: Imputar coordenadas finales faltantes")
nulls_end_lat = df['End_Lat'].isnull().sum()
nulls_end_lng = df['End_Lng'].isnull().sum()
df['End_Lat'] = df['End_Lat'].fillna(df['Start_Lat'])
df['End_Lng'] = df['End_Lng'].fillna(df['Start_Lng'])
print(f"   End_Lat: {nulls_end_lat:,} -> {df['End_Lat'].isnull().sum():,} nulos")
print(f"   End_Lng: {nulls_end_lng:,} -> {df['End_Lng'].isnull().sum():,} nulos")


PASO 8: Imputar coordenadas finales faltantes
   End_Lat: 236,182 -> 0 nulos
   End_Lng: 236,182 -> 0 nulos


In [68]:
# PASO 9: Imputar Precipitation con 0 (sin reporte = sin lluvia)
print("\nPASO 9: Imputar precipitacion faltante")
nulls_precip = df['Precipitation(in)'].isnull().sum()
df['Precipitation(in)'] = df['Precipitation(in)'].fillna(0)
print(f"   Precipitation(in): {nulls_precip:,} -> {df['Precipitation(in)'].isnull().sum():,} nulos")


PASO 9: Imputar precipitacion faltante
   Precipitation(in): 64,475 -> 0 nulos


In [69]:
# PASO 10: Imputar Wind_Chill con Temperature
print("\nPASO 10: Imputar sensacion termica faltante")
nulls_wind_chill = df['Wind_Chill(F)'].isnull().sum()
df['Wind_Chill(F)'] = df['Wind_Chill(F)'].fillna(df['Temperature(F)'])
print(f"   Wind_Chill(F): {nulls_wind_chill:,} -> {df['Wind_Chill(F)'].isnull().sum():,} nulos")


PASO 10: Imputar sensacion termica faltante
   Wind_Chill(F): 54,508 -> 0 nulos


In [70]:
# PASO 11: Imputar Wind_Direction con moda por estado
print("\nPASO 11: Imputar direccion del viento faltante")
nulls_wind_dir = df['Wind_Direction'].isnull().sum()
df['Wind_Direction'] = df.groupby('State')['Wind_Direction'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'N')
)
print(f"   Wind_Direction: {nulls_wind_dir:,} -> {df['Wind_Direction'].isnull().sum():,} nulos")


PASO 11: Imputar direccion del viento faltante
   Wind_Direction: 48,385 -> 0 nulos


In [71]:
# PASO 12: Imputar Weather_Timestamp con Start_Time
print("\nPASO 12: Imputar timestamp climatico faltante")
nulls_weather_ts = df['Weather_Timestamp'].isnull().sum()
df['Weather_Timestamp'] = df['Weather_Timestamp'].fillna(df['Start_Time'])
print(f"   Weather_Timestamp: {nulls_weather_ts:,} -> {df['Weather_Timestamp'].isnull().sum():,} nulos")


PASO 12: Imputar timestamp climatico faltante
   Weather_Timestamp: 29,595 -> 0 nulos


In [80]:
df['Start_Time'] = pd.to_datetime(df['Start_Time'], errors='coerce')

In [81]:
# PASO 13: Imputar columnas Twilight con moda por hora del dia
print("\nPASO 13: Imputar columnas astronomicas faltantes")
df['Hour'] = df['Start_Time'].dt.hour
twilight_cols = ['Sunrise_Sunset', 'Nautical_Twilight', 'Civil_Twilight', 'Astronomical_Twilight']
for col in twilight_cols:
    if col in df.columns:
        nulls_antes = df[col].isnull().sum()
        df[col] = df.groupby('Hour')[col].transform(
            lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Day')
        )
        print(f"   {col}: {nulls_antes:,} -> {df[col].isnull().sum():,} nulos")
df.drop('Hour', axis=1, inplace=True)


PASO 13: Imputar columnas astronomicas faltantes
   Sunrise_Sunset: 14,896 -> 0 nulos
   Nautical_Twilight: 14,896 -> 0 nulos
   Civil_Twilight: 14,896 -> 0 nulos
   Astronomical_Twilight: 14,896 -> 0 nulos


In [73]:
# PASO 14: Imputar Weather_Condition faltante
print("\nPASO 14: Imputar condicion climatica faltante")
nulls_weather = df['Weather_Condition'].isnull().sum()
df['Weather_Condition'] = df['Weather_Condition'].fillna('No information')
df['Weather_Condition'] = df['Weather_Condition'].str.strip().str.title()
print(f"   Weather_Condition: {nulls_weather:,} -> {df['Weather_Condition'].isnull().sum():,} nulos")


PASO 14: Imputar condicion climatica faltante
   Weather_Condition: 38,688 -> 0 nulos


In [74]:
# PASO 15: Imputar City con moda por estado
print("\nPASO 15: Imputar ciudad faltante")
nulls_city = df['City'].isnull().sum()
df['City'] = df.groupby('State')['City'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Unknown')
)
print(f"   City: {nulls_city:,} -> {df['City'].isnull().sum():,} nulos")


PASO 15: Imputar ciudad faltante
   City: 65 -> 0 nulos


In [75]:
# PASO 16: Imputar Zipcode con moda por ciudad
print("\nPASO 16: Imputar codigo postal faltante")
nulls_zip = df['Zipcode'].isnull().sum()
df['Zipcode'] = df.groupby('City')['Zipcode'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else '00000')
)
print(f"   Zipcode: {nulls_zip:,} -> {df['Zipcode'].isnull().sum():,} nulos")


PASO 16: Imputar codigo postal faltante
   Zipcode: 307 -> 0 nulos


In [76]:
# PASO 17: Imputar Timezone con moda por estado
print("\nPASO 17: Imputar zona horaria faltante")
nulls_tz = df['Timezone'].isnull().sum()
df['Timezone'] = df.groupby('State')['Timezone'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'US/Eastern')
)
print(f"   Timezone: {nulls_tz:,} -> {df['Timezone'].isnull().sum():,} nulos")


PASO 17: Imputar zona horaria faltante
   Timezone: 1,713 -> 0 nulos


In [77]:
# PASO 18: Imputar Street con 'No information'
print("\nPASO 18: Imputar calle faltante")
nulls_street = df['Street'].isnull().sum()
df['Street'] = df['Street'].fillna('No information')
print(f"   Street: {nulls_street:,} -> {df['Street'].isnull().sum():,} nulos")


PASO 18: Imputar calle faltante
   Street: 7,236 -> 0 nulos


In [78]:
# PASO 19: Imputar Airport_Code con 'No information'
print("\nPASO 19: Imputar codigo de aeropuerto faltante")
nulls_airport = df['Airport_Code'].isnull().sum()
df['Airport_Code'] = df['Airport_Code'].fillna('No information')
print(f"   Airport_Code: {nulls_airport:,} -> {df['Airport_Code'].isnull().sum():,} nulos")


PASO 19: Imputar codigo de aeropuerto faltante
   Airport_Code: 7,199 -> 0 nulos


In [ ]:
# PASO 21: Imputar End_Time con Start_Time + Duracion promedio
print("\nPASO 20: Imputar End_Time faltante")
df['Duration'] = (df['End_Time'] - df['Start_Time']).dt.total_seconds() / 60
avg_duration = df['Duration'].mean()
nulls_end_time = df['End_Time'].isnull().sum()
df.loc[df['End_Time'].isnull(), 'End_Time'] = df['Start_Time'] + pd.to_timedelta(avg_duration, unit='m')
print(f"   End_Time: {nulls_end_time:,} -> {df['End_Time'].isnull().sum():,} nulos")


PASO 20: Imputar End_Time faltante
   End_Time: 515,000 -> 0 nulos


In [89]:
df.drop(columns=['Duration'], inplace=True)

In [91]:
# PASO 22: Verificar valores nulos restantes
print("\nPASO 20: Verificacion final de valores nulos")
print("="*70)
remaining_nulls = df.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0].sort_values(ascending=False)
if len(remaining_nulls) > 0:
    print("Columnas con valores nulos restantes:")
    for col, count in remaining_nulls.items():
        pct = (count / len(df) * 100)
        print(f"   {col}: {count:,} ({pct:.2f}%)")
else:
    print("No hay valores nulos restantes en el dataset")
print("="*70)


PASO 20: Verificacion final de valores nulos
No hay valores nulos restantes en el dataset


In [92]:
# PASO 20: Guardar dataset limpio
df.to_csv('accidents_2022_clean.csv', index=False)
print("\n" + "="*70)
print("LIMPIEZA COMPLETADA")
print(f"   Dataset limpio guardado: accidents_2022_clean.csv")
print(f"   Registros finales: {len(df):,}")
print(f"   Reduccion total: {(1 - len(df)/total_records)*100:.2f}%")
print("="*70)


LIMPIEZA COMPLETADA
   Dataset limpio guardado: accidents_2022_clean.csv
   Registros finales: 1,762,449
   Reduccion total: 0.00%


In [93]:
print("\n" + "="*70)
print("🔧 CREACIÓN DE NUEVAS MÉTRICAS")
print("="*70)

# Feature 1: Componentes temporales
print("\n1. COMPONENTES TEMPORALES:")
df['Hour'] = df['Start_Time'].dt.hour
df['Day_of_Week'] = df['Start_Time'].dt.dayofweek  # 0=Lunes, 6=Domingo
df['Day_Name'] = df['Start_Time'].dt.day_name()
df['Month'] = df['Start_Time'].dt.month
df['Month_Name'] = df['Start_Time'].dt.month_name()
df['Is_Weekend'] = df['Day_of_Week'].isin([5, 6]).astype(int)
print(f"    Hour (0-23)")
print(f"    Day_of_Week (0-6)")
print(f"    Day_Name (nombre del día)")
print(f"    Month (1-12)")
print(f"    Month_Name (nombre del mes)")
print(f"    Is_Weekend (0/1)")



🔧 CREACIÓN DE NUEVAS MÉTRICAS

1. COMPONENTES TEMPORALES:
    Hour (0-23)
    Day_of_Week (0-6)
    Day_Name (nombre del día)
    Month (1-12)
    Month_Name (nombre del mes)
    Is_Weekend (0/1)


In [ ]:
# Feature 2: Período del día
print("\n2. PERÍODO DEL DÍA:")
def classify_time_period(hour):
    if 6 <= hour < 12: return 'Mañana'
    elif 12 <= hour < 18: return 'Tarde'
    elif 18 <= hour < 22: return 'Noche'
    else: return 'Madrugada'
    
df['Periodo_Dia'] = df['Hour'].apply(classify_time_period)
df['Is_Rush_Hour'] = df['Hour'].isin([7,8,9,17,18,19]).astype(int)
print(f"    Periodo_Dia: {df['Periodo_Dia'].unique()}")
print(f"    Is_Rush_Hour (hora pico: 7-9, 17-19)")



2. PERÍODO DEL DÍA:
    Periodo_Dia: ['Madrugada' 'Mañana' 'Tarde' 'Noche']
    Is_Rush_Hour (hora pico: 7-9, 17-19)
    Ref: Chen et al. (2016) - Spatio-temporal impacts


In [95]:
# Feature 3: Frecuencia de accidentes por ubicación
print("\n3. FRECUENCIA POR UBICACIÓN:")
city_counts = df.groupby('City').size()
df['City_Accident_Count'] = df['City'].map(city_counts)
state_counts = df.groupby('State').size()
df['State_Accident_Count'] = df['State'].map(state_counts)
print(f"    City_Accident_Count (rango: {df['City_Accident_Count'].min()}-{df['City_Accident_Count'].max()})")
print(f"    State_Accident_Count (rango: {df['State_Accident_Count'].min()}-{df['State_Accident_Count'].max()})")



3. FRECUENCIA POR UBICACIÓN:
    City_Accident_Count (rango: 1-64545)
    State_Accident_Count (rango: 37-375913)


In [96]:
# Feature 4: Condición climática simplificada
print("\n4. CATEGORIZACIÓN CLIMÁTICA SIMPLIFICADA:")
def simplify_weather(condition):
    condition = str(condition).lower()
    if 'clear' in condition or 'fair' in condition:
        return 'Despejado'
    elif 'cloud' in condition or 'overcast' in condition:
        return 'Nublado'
    elif 'rain' in condition or 'drizzle' in condition:
        return 'Lluvia'
    elif 'snow' in condition or 'ice' in condition or 'sleet' in condition:
        return 'Nieve/Hielo'
    elif 'fog' in condition or 'mist' in condition:
        return 'Niebla'
    elif 'thunder' in condition or 'storm' in condition:
        return 'Tormenta'
    else:
        return 'Otros'

df['Weather_Category'] = df['Weather_Condition'].apply(simplify_weather)
print(f"    Weather_Category: {df['Weather_Category'].unique()}")

# Guardar con nuevas features
df.to_csv('accidents_2022_clean.csv', index=False)
print(f"\n Métricas creadas y guardadas")


4. CATEGORIZACIÓN CLIMÁTICA SIMPLIFICADA:
    Weather_Category: ['Despejado' 'Nublado' 'Otros' 'Niebla' 'Lluvia' 'Tormenta' 'Nieve/Hielo']

 Métricas creadas y guardadas
